# 15. Testing

Testing is not a luxury for senior engineers. It is a core engineering practice that protects correctness, makes refactors safe, and gives confidence when production systems change.

This notebook covers three layers:
- Unit testing: fast feedback on individual behavior
- Integration testing: verifying components work together
- Advanced testing: contracts, properties, resilience, and end-to-end confidence

A good testing strategy is not about testing everything by mocking everything. The best tests validate real behavior and only mock boundaries that are external or non-deterministic.


## 1. Why Testing Matters

Testing creates safety nets.

Without tests, engineers often rely on manual checks, and those checks miss edge cases. Good tests let you:
- catch regressions early
- document intended behavior
- refactor with confidence
- communicate contracts across teams

The key idea is not "more tests" for its own sake. It is "the right tests" that protect real behavior.


In [1]:
def add(a, b):
    return a + b

assert add(2, 3) == 5
assert add(-1, 1) == 0

print("basic assertions passed")

# Theory:
# - assertions are the simplest form of tests
# - they detect when code deviates from expected behavior
# - they are useful, but not enough for larger systems


basic assertions passed


## 2. pytest

pytest is the standard test framework in Python.

It is simple to write, expressive to read, and excellent for unit tests, integration tests, and structured test suites.

Key ideas:
- tests are plain Python functions
- naming conventions matter: functions start with `test_`
- assertions are the core mechanism
- fixtures provide setup and teardown


In [2]:
def multiply(a, b):
    return a * b


def test_multiply_basic():
    assert multiply(3, 4) == 12


def test_multiply_zero():
    assert multiply(5, 0) == 0


print("pytest-style tests defined")

# Theory:
# - pytest discovers test functions automatically
# - assertions express expected behavior clearly
# - the function name tells pytest this is a test case


pytest-style tests defined


## 3. Fixtures

Fixtures are reusable test setup objects.

They help you avoid duplicated setup code and keep tests readable. Common uses include:
- test database setup
- temporary files/directories
- mock services
- application client creation

A fixture is often created with the `@pytest.fixture` decorator.


In [4]:
%pip install pytest

import pytest

@pytest.fixture
def sample_data():
    return {"items": [1, 2, 3], "status": "ok"}


def test_fixture_usage(sample_data):
    assert sample_data["status"] == "ok"
    assert len(sample_data["items"]) == 3

print("fixture example executed")

# Theory:
# - fixtures centralize setup and teardown
# - they improve reuse and reduce duplication
# - they make tests easier to maintain and understand


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


fixture example executed


## 4. Parametrization

Parametrization lets you run the same test with many different values.

This is ideal for validating boundary conditions and multiple valid inputs without writing repetitive tests.


In [5]:
import pytest


def is_even(value):
    return value % 2 == 0


@pytest.mark.parametrize(
    "value, expected",
    [
        (2, True),
        (3, False),
        (0, True),
        (-4, True),
    ],
)
def test_is_even(value, expected):
    assert is_even(value) is expected

print("parameterized examples prepared")

# Theory:
# - the same test logic runs against multiple inputs
# - it helps cover boundaries and edge cases efficiently
# - it reduces boilerplate and makes regressions easier to spot


parameterized examples prepared


## 5. Mocking and unittest.mock

Mocks are useful when the code under test interacts with external systems or when you want to isolate a unit from dependencies.

The important rule: mock at boundaries, not at the center of the business logic.

Python's `unittest.mock` library gives you tools like `Mock`, `MagicMock`, `patch`, and `call`.


In [6]:
from unittest.mock import Mock, patch

class PaymentService:
    def charge(self, amount):
        return f"charged {amount}"

class OrderService:
    def __init__(self, payment_service):
        self.payment_service = payment_service

    def checkout(self, total):
        return self.payment_service.charge(total)


mock_payment = Mock()
mock_payment.charge.return_value = "charged 99"
order = OrderService(mock_payment)
print(order.checkout(99))

with patch('__main__.PaymentService.charge', return_value='mocked charge') as mocked_charge:
    service = PaymentService()
    print(service.charge(42))
    print(mocked_charge.called)

# Theory:
# - Mock objects simulate external dependencies
# - patch replaces methods or functions temporarily at runtime
# - this is valuable for network, database, and API boundaries
# - do not mock internal business rules unless that is the actual boundary


charged 99
mocked charge
True


## 6. Monkeypatching and Test Isolation

Monkeypatching modifies objects or functions during a test.

This is useful for replacing environment variables, dependencies, or runtime configuration in a controlled way.

Test isolation means each test should run independently and avoid leaking state into other tests.


In [7]:
import os

def get_env_value():
    return os.getenv("APP_MODE", "dev")


def test_monkeypatch(monkeypatch):
    monkeypatch.setenv("APP_MODE", "prod")
    assert get_env_value() == "prod"


def test_isolation_example():
    assert get_env_value() in {"dev", "prod"}

print("monkeypatch and isolation example executed")

# Theory:
# - monkeypatch modifies behavior only inside a test
# - it is a controlled way to simulate runtime configuration
# - tests should not depend on previous tests polluting state


monkeypatch and isolation example executed


## 7. What Not to Mock

Over-mocking makes tests weak.

Bad tests often mock the very logic being validated. That means the test passes while the system still fails in production.

Avoid mocking:
- core business rules
- pure functions whose behavior is easy to test directly
- data transformations that are part of your real logic
- real domain objects where simple objects are enough

Prefer mocks at the edges:
- network calls
- database drivers
- external APIs
- file systems when you are not testing file behavior itself

A good rule: if the behavior is important and local, test the real implementation.


In [8]:
def total_after_tax(subtotal, tax_rate):
    return subtotal * (1 + tax_rate)

# Good test: real logic, no mocking
assert abs(total_after_tax(100, 0.2) - 120.0) < 1e-9

# Avoid this anti-pattern:
# mocking the function itself instead of testing the real calculation

print("real behavior validated directly")

# Theory:
# - tests should validate actual domain logic
# - mocks should represent boundaries, not core implementation
# - over-mocking creates false confidence


real behavior validated directly


## 8. Integration Testing

Unit tests check individual components in isolation.

Integration tests check that components work together in realistic scenarios.

Examples include:
- database tests
- API tests
- Kafka tests
- Redis tests
- external API testing

The goal is to validate collaborations, not just isolated functions.


In [9]:
from dataclasses import dataclass

@dataclass
class Order:
    id: int
    status: str

class OrderRepository:
    def __init__(self):
        self._orders = {}

    def save(self, order):
        self._orders[order.id] = order
        return order

    def get(self, order_id):
        return self._orders.get(order_id)


def test_repository_round_trip():
    repo = OrderRepository()
    order = Order(id=1, status="pending")
    repo.save(order)
    assert repo.get(1).status == "pending"

print("repository integration-style example passed")

# Theory:
# - integration tests exercise real collaboration between components
# - they often use in-memory or test databases to mimic production interactions
# - this type of test catches wiring and contract issues that unit tests miss


repository integration-style example passed


## 9. API Testing

API tests validate the boundary where clients talk to your services.

They check:
- request parsing
- validation
- response codes
- authentication behavior
- contract stability

In Python, FastAPI apps are often tested with pytest and TestClient.


In [11]:
%pip install httpx

from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok"}

client = TestClient(app)

response = client.get("/health")
print(response.status_code)
print(response.json())

# Theory:
# - API tests check how external clients interact with the service
# - they validate serialization, validation, routing, and status codes
# - this is closer to real production behavior than pure function tests


  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
Note: you may need to restart the kernel to use updated packages.
200
{'status': 'ok'}



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
d:\Documents\Python\Python-Complete-Guide\python\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 10. Database, Kafka, and Redis Tests

Many real-world systems depend on external systems such as:
- PostgreSQL / MySQL / SQLite for persistence
- Kafka for event streaming
- Redis for cache and pub/sub behavior

These tests often require containerized or test environments, because you want to validate integration behavior without relying on the production stack.

Examples of good tests:
- save and read a record from a test database
- produce and consume a Kafka message
- set and read a Redis value

These are integration tests, not pure unit tests.


In [12]:
import sqlite3

conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.execute('CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)')
cur.execute('INSERT INTO users (name) VALUES (?)', ('Alice',))
cur.execute('SELECT name FROM users WHERE id = 1')
print(cur.fetchone())
conn.close()

# Theory:
# - this is a lightweight database integration example
# - real database tests often use test databases or containers
# - the principle is the same: validate actual persistence behavior


('Alice',)


## 11. External API Testing

External API testing is necessary when your system depends on third-party services.

This is different from mocking everything.

You may test:
- HTTP status and payload shape
- retry logic
- timeout behavior
- invalid response handling
- data contract compatibility

Many teams use recorded responses, contract tests, or sandbox environments for this.


In [14]:
%pip install requests

import requests

# This is an example of validating a real HTTP call pattern.
# In production, this should be a controlled sandbox or mock server.
url = 'https://httpbin.org/get'
response = requests.get(url, timeout=10)
print(response.status_code)
print(response.json()['url'])

# Theory:
# - external API testing checks real HTTP integration behavior
# - it must also validate resilience against timeouts and bad payloads
# - production-safe tests often use sandbox environments or recorded responses


  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.1-cp313-cp313-win_amd64.whl.metadata (46 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.1-cp313-cp313-win_amd64.whl (199 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


200
https://httpbin.org/get


## 12. Contract Testing

Contract testing verifies that two systems agree on the structure and meaning of a contract.

This is common for:
- provider/consumer API contracts
- event schemas
- payload compatibility between services

If the contract changes, tests fail before production breaks downstream systems.


In [15]:
from pydantic import BaseModel, ValidationError

class UserContract(BaseModel):
    id: int
    name: str
    email: str

payload = {"id": 10, "name": "Alice", "email": "alice@example.com"}
user = UserContract(**payload)
print(user.model_dump())

try:
    UserContract(id='bad', name='Bob', email='bob@example.com')
except ValidationError as exc:
    print(type(exc).__name__)
    print(exc.errors())

# Theory:
# - contract tests validate the agreed payload shape
# - they detect drift between producer and consumer expectations
# - this is especially important in distributed systems


{'id': 10, 'name': 'Alice', 'email': 'alice@example.com'}
ValidationError
[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'bad', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}]


## 13. Property-Based Testing

Property-based testing checks general rules instead of a few handcrafted examples.

Instead of writing one test for each input, you define a property that should always hold.

Examples:
- for any list, sorting it twice yields the same result
- for any integer, `x + 0 == x`
- for any positive number, `abs(x) >= 0`

Hypothesis is the popular Python library for property-based testing.


In [17]:
%pip install hypothesis

from hypothesis import given, strategies as st

@given(st.lists(st.integers()))
def test_sort_is_idempotent(values):
    sorted_twice = sorted(sorted(values))
    assert sorted(values) == sorted_twice

print("property-based test example prepared")

# Theory:
# - property-based tests generate many inputs automatically
# - they find edge cases that manual examples often miss
# - they are excellent for validating invariants and general rules


   ---------------------------------------- 0.0/672.3 kB ? eta -:--:--
   --------------------------------------- 672.3/672.3 kB 12.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
property-based test example prepared



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 14. Load, Stress, and Chaos Testing

These tests evaluate performance and resilience.

- Load testing: how a system behaves under expected traffic
- Stress testing: how it behaves under extreme conditions
- Chaos testing: how it reacts when dependencies fail or become slow

These are often run in staging or production-like environments, and they are essential for systems with reliability requirements.


In [18]:
import time
import random


def service_call():
    time.sleep(0.05)
    return "ok"


def run_load_sample(count=20):
    start = time.time()
    results = []
    for _ in range(count):
        results.append(service_call())
    elapsed = time.time() - start
    print(f"processed {count} calls in {elapsed:.3f}s")
    return results


run_load_sample()

# Theory:
# - load tests measure throughput and latency under realistic concurrency
# - stress tests push beyond expected capacity
# - chaos tests simulate failures so recovery behavior can be validated


processed 20 calls in 1.007s


['ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok',
 'ok']

## 15. End-to-End Testing

End-to-end tests exercise the system the way a user or another service experiences it.

This includes:
- browser flows
- UI actions
- API requests through the real stack
- deployment and configuration checks

E2E tests are slower and more expensive, but they catch integration failures that unit tests and even many API tests miss.


In [19]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {"id": user_id, "name": "Alice"}

client = TestClient(app)
response = client.get('/users/42')
print(response.status_code)
print(response.json())

# Theory:
# - E2E tests simulate the real path a user or service takes through the system
# - they validate routing, serialization, business logic, and responses together
# - they are slower, but they provide the highest confidence for critical flows


200
{'id': 42, 'name': 'Alice'}


## 16. Test Strategy Summary

A strong engineering team does not rely on only one test type.

The right mix is:
- unit tests for fast validation of logic
- integration tests for component collaboration
- contract tests for service boundaries
- E2E tests for critical flows
- resilience tests for systems under failure and load

Testing is not just checking whether code works. It is checking whether the system remains safe as it evolves.
